In [1]:
import os
import torch
import queue
import threading
import numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModel

In [2]:
def extract_worker(work_queue, output_npy_folder, model_id, batch_size, gpu_id):
    """
    Luồng xử lý (Worker): Rút thư mục từ Queue ra xử lý cho đến khi Queue rỗng.
    """
    device = f"cuda:{gpu_id}"
    print(f"[GPU {gpu_id}] Đang tải model trên {device}...")

    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id, dtype=torch.float16)
    model = model.to(device).half()  # Thêm .half() để dứt khoát ép toàn bộ sang float16
    model.eval()

    while True:
        try:
            video_dir = work_queue.get(timeout=3)
        except queue.Empty:
            print(f"[GPU {gpu_id}] Hàng đợi trống. Đã hoàn thành công việc và thoát.")
            break

        video_name = os.path.basename(video_dir)
        output_npy_path = os.path.join(output_npy_folder, f"{video_name}.npy")

        if os.path.exists(output_npy_path):
            print(f"[GPU {gpu_id}] -> Đã tồn tại {video_name}.npy, bỏ qua... (Còn lại: {work_queue.qsize()})")
            work_queue.task_done() 
            continue

        image_files = sorted([f for f in os.listdir(video_dir) if f.endswith(('.jpg', '.png'))])
        if not image_files:
            work_queue.task_done()
            continue

        print(f"[GPU {gpu_id}] Đang xử lý: {video_name} ({len(image_files)} ảnh) | Hàng đợi còn: {work_queue.qsize()}")
        video_features = []

        for i in range(0, len(image_files), batch_size):
            batch_files = image_files[i:i+batch_size]
            images = []
            for img_file in batch_files:
                try:
                    images.append(Image.open(os.path.join(video_dir, img_file)).convert("RGB"))
                except:
                    pass
            if not images: continue

            inputs = processor(images=images, return_tensors="pt").to(device)

            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
                features = model.get_image_features(**inputs)
                
                if hasattr(features, 'pooler_output'):
                    features = features.pooler_output
                elif hasattr(features, 'image_embeds'):
                    features = features.image_embeds

                features = features / features.norm(p=2, dim=-1, keepdim=True)
                video_features.append(features.to(torch.float32).cpu().numpy())

        if video_features:
            video_features_np = np.vstack(video_features)
            np.save(output_npy_path, video_features_np)
            print(f"[GPU {gpu_id}]   [OK] Đã lưu {video_name}.npy | Shape: {video_features_np.shape}")

        work_queue.task_done()

def extract_siglip_features(list_video_dirs, output_npy_folder, model_id='google/siglip2-so400m-patch16-naflex', batch_size=32):
    """
    Hàm chính: Nạp việc vào Queue và quản lý các luồng.
    """
    os.makedirs(output_npy_folder, exist_ok=True)
    
    num_gpus = torch.cuda.device_count()
    if num_gpus == 0:
        print("Lỗi: Không tìm thấy GPU nào!")
        return

    work_queue = queue.Queue()
    for video_dir in list_video_dirs:
        work_queue.put(video_dir)

    print(f"✅ Đã nạp {work_queue.qsize()} thư mục vào Queue. Phát hiện {num_gpus} GPU. Khởi chạy {num_gpus} workers...")

    threads = []
    for gpu_id in range(num_gpus):
        t = threading.Thread(
            target=extract_worker, 
            args=(work_queue, output_npy_folder, model_id, batch_size, gpu_id)
        )
        t.start()
        threads.append(t)
        
    work_queue.join()
    for t in threads:
        t.join()

    print("🎉 HOÀN TẤT TRÍCH XUẤT TRÊN TẤT CẢ GPU!")

In [7]:
import glob

all_video_dirs = glob.glob("/kaggle/input/datasets/bumbleboo/aic26-b2-taylor/dataset/Keyframes_*/*")

extract_siglip_features(
    list_video_dirs=all_video_dirs, 
    output_npy_folder="/kaggle/working/siglip_features"
)

✅ Đã nạp 2 thư mục vào Queue. Phát hiện 2 GPU. Khởi chạy 2 workers...
[GPU 0] Đang tải model trên cuda:0...
[GPU 1] Đang tải model trên cuda:1...


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

[GPU 1] Đang xử lý: L21_V001 (1013 ảnh) | Hàng đợi còn: 1
[GPU 0] Đang xử lý: L21_V002 (846 ảnh) | Hàng đợi còn: 0
[GPU 0]   [OK] Đã lưu L21_V002.npy | Shape: (846, 1152)
[GPU 0] Hàng đợi trống. Đã hoàn thành công việc và thoát.
[GPU 1]   [OK] Đã lưu L21_V001.npy | Shape: (1013, 1152)
[GPU 1] Hàng đợi trống. Đã hoàn thành công việc và thoát.
🎉 HOÀN TẤT TRÍCH XUẤT TRÊN TẤT CẢ GPU!


In [4]:
!zip -0 -r '/kaggle/working/siglip_features.zip' '/kaggle/working/siglip_features'

	zip warning: name not matched: /kaggle/working/siglip_features

zip error: Nothing to do! (try: zip -0 -r /kaggle/working/siglip_features.zip . -i /kaggle/working/siglip_features)


In [5]:
from IPython.display import FileLink

# Tạo link tải trực tiếp
FileLink(r'siglip_features.zip')

/kaggle/working/siglip_features.zip